### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
    rating_threshold=4.0, # replica 3.5
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75, # replica 0.64
    val_ratio=0.10, # replica 0.16
    test_ratio=0.15, # replica 0.20
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...


Building labels...
Interaction Graph: Data(edge_index=[2, 317967], edge_label=[158984])
Edge Index: tensor([[    0,     0,     0,  ..., 10758, 10762, 10763],
        [ 2089,  2202,  2318,  ...,  1760,  1645,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True, rescale=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:05<00:00, 348.67it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, ...",0.370445,"[0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, ...",0.292809,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.480054,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.645735


#### Prepare Item Multihot Vec on Each Dimension

In [9]:
item_vec_df = evaluator.get_item_feature_multihot_vec(encoded_train_df, feature_engineer.vocab2idx)
item_vec_df.head(1)

,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,1588,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


#### Prepare train/valid triplet data

In [10]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df = train_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="left")
# train_triplet_with_dps_df.head(1)

valid_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_valid_df, k_negative_samples=5)
valid_triplet_with_dps_df = valid_triplet_df.merge(user_dps_df, on="userID", how="left")
valid_triplet_with_dps_df = valid_triplet_with_dps_df.merge(item_vec_df, left_on="pos_item_id", right_on="movieID", how="inner")
valid_triplet_with_dps_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920
Original data count (positive samples): 18261
Num of triplets: 18261(pos samples) * 5(negative sampled items) = 91305


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,...,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps,movieID,actorID_vec,country_vec,directorID_vec,genre_vec
0,0,962,4662,"[328, 704, 1, 1, 1]",37,464,"[2, 3, 6, 10, 12, 0, 0, 0]","[988, 648, 2087, 1, 924]",37,75,...,0.292809,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.480054,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.645735,962,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ..."


#### Prepare prediction pool for inference/testing

In [11]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=1000)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 1000 items for each user
Num of interactions: 2064(users) * 1000(items) = 2064000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
2063995,2063,1660,0,"[1, 735, 1165, 1, 1]",37,1,"[4, 5, 0, 0, 0, 0, 0, 0]"
2063996,2063,2347,0,"[1, 1, 1, 1, 1]",18,134,"[4, 5, 0, 0, 0, 0, 0, 0]"
2063997,2063,4479,0,"[867, 884, 416, 2282, 1]",37,373,"[6, 17, 0, 0, 0, 0, 0, 0]"
2063998,2063,5256,0,"[865, 1756, 1810, 1, 1]",37,358,"[9, 0, 0, 0, 0, 0, 0, 0]"
2063999,2063,78,0,"[1, 1356, 1, 1, 1]",37,1,"[9, 16, 0, 0, 0, 0, 0, 0]"


### Prepare DataLoader

In [12]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset, UserPosItemSampler, get_user_triplet_mapping

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
valid_dataset = TripletDataset(valid_triplet_with_dps_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

# TODO: Custom sampler
# MIN_POS_ITEMS = 2
# user_to_pos_items, user_pos_to_indices = get_user_triplet_mapping(train_triplet_with_dps_df, MIN_POS_ITEMS)
# train_sampler = UserPosItemSampler(user_to_pos_items, user_pos_to_indices, batch_size=BATCH_SIZE, min_pos_items=MIN_POS_ITEMS, max_pos_items=20, buffer=0)
# train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=4, worker_init_fn=seed_worker, generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 90805
test data count: 2064000


### Configure Model (LightningModule)

In [13]:
from lightning_models.mtdp_ngcf_v2 import MTDPRecSRM

EMB_DIM = 64
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

DPR_WEIGHTS = {
    "actor_dpr": 0.25,
    "country_dpr": 0.25,
    "director_dpr": 0.25,
    "genre_dpr": 0.25,
}

DPM_WEIGHTS = {
    "actor_pd": 0.25,
    "country_pd": 0.25,
    "director_pd": 0.25,
    "genre_pd": 0.25,
}

MT_WEIGHTS = {
    "bpr_loss": 1.0,
    "dps_loss": 0,
    "dpr_loss": 0,
    "dpm_loss": 0,
    "l2_loss": 0,
}

RESCALE_METHOD = None  # None, "log", "ema"

model = MTDPRecSRM(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    dpr_weights=DPR_WEIGHTS,
    dpm_weights=DPM_WEIGHTS,
    mt_weights=MT_WEIGHTS,
    rescale_method=RESCALE_METHOD,
)


Seed set to 42


### Configure Trainer and Experiment

In [14]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "rerank"
VERSION = "ngcf_v2"
RUN_NAME = "run02(K=1000)"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    version_name=VERSION,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_loss",
    monitor_mode="min",
    hyper_param_str=f"",
)

In [15]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [16]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/experiments/test_checkpoints/rerank exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

   | Name                | Type              | Params | Mode 
-------------------------------------------------------------------
0  | ngcf_model          | NGCF              | 714 K  | train
1  | bpr_loss            | BPRLoss           | 0      | train
2  | reg_loss            | EmbLoss           | 0      | train
3  | dps_module          | DPSPredictor      | 1.0 K  | train
4  | dps_loss_fn         | DPSLoss           | 0      | train
5  | dpr_module          | DPRegularizer     | 263 K  | train
6  | dpr_loss_fn         | DPRLoss           | 0      | train
7  | dpm_module          | DPMatcher         | 0      | train
8  | dpm_loss_fn         | KLDivergenceLoss  | 0      | train
9  | loss_scaling_module | LogScaleLoss  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 3885: 'val_loss' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_loss did not improve in the last 5 records. Best score: 0.595. Signaling Trainer to stop.
Epoch 5, global step 4662: 'val_loss' was not in top 1


🏃 View run run02(K=1000) at: http://140.112.106.216:3683/#/experiments/19/runs/b076cafe2cff43aba8e8ff24010e2756
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/19


### Inference

In [17]:
# NOTE: the inference model MUST be the same as the training model
# best_model_experiment_name = "rerank"
# best_model_checkpoint_path = "[ngcf_v2]-test_run--best-checkpoint-epoch=00-val_loss=2.93.ckpt"
# best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"
best_model_path = trainer.checkpoint_callback.best_model_path

model = MTDPRecSRM.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.2665884494781494     │
│        test_ndcg20        │    0.30366259813308716    │
│        test_ndcg5         │    0.21629247069358826    │
│     test_precision10      │    0.08667635917663574    │
│     test_precision20      │    0.07628391683101654    │
│      test_precision5      │    0.09127906709909439    │
│       test_recall10       │    0.07743934541940689    │
│       test_recall20       │    0.1295858472585678     │
│       test_recall5        │   0.042407888919115067    │
└───────────────────────────┴───────────────────────────┘

🏃 View run run02(K=1000) at: http://140.112.106.216:3683/#/experiments/19/runs/b076cafe2cff43aba8e8ff24010e2756
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/19


[{'test_ndcg5': 0.21629247069358826,
  'test_ndcg10': 0.2665884494781494,
  'test_ndcg20': 0.30366259813308716,
  'test_precision5': 0.09127906709909439,
  'test_precision10': 0.08667635917663574,
  'test_precision20': 0.07628391683101654,
  'test_recall5': 0.042407888919115067,
  'test_recall10': 0.07743934541940689,
  'test_recall20': 0.1295858472585678}]

In [18]:
# exp 1000
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.216292,0.042408,0.091279,0.266588,0.077439,0.086676,0.303663,0.129586,0.076284
std,595.969798,0.325612,0.093088,0.146179,0.302242,0.128498,0.112357,0.269284,0.164335,0.084331
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1031.500000,0.000000,0.000000,0.000000,0.289065,0.015152,0.100000,0.308083,0.083333,0.050000
75%,1547.250000,0.430677,0.055556,0.200000,0.483813,0.111111,0.100000,0.481450,0.200000,0.100000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.700000


In [19]:
# model.test_results["eval_score_df"].describe()

In [20]:
# Evaluate user diversity preference matching score (DPMS) at k
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)

# Get user DPMS
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 724.13it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.140615,0.958347,0.142554,0.838152,0.519917
std,595.969798,0.077288,0.053022,0.107839,0.089421,0.058149
min,0.000000,0.000000,0.353897,0.000000,0.353746,0.291433
25%,515.750000,0.080195,0.950464,0.055482,0.798360,0.482091
50%,1031.500000,0.137590,0.975951,0.132499,0.856891,0.521950
75%,1547.250000,0.195790,0.987844,0.215539,0.903169,0.561195
max,2063.000000,0.440447,1.000000,0.533463,0.980082,0.684934


In [21]:
prefix = "(mt)ngcf_v2_k5_1000"
embedding_path = "embeddings/"

torch.save(model.user_emb.cpu(), f"{embedding_path}{prefix}_user_emb.pt")
torch.save(model.item_emb.cpu(), f"{embedding_path}{prefix}_item_emb.pt")

In [22]:
import joblib
dir = "artifacts"
prefix = "(mt)ngcf_v2_k5_1000"
if not os.path.exists(dir):
    os.makedirs(dir)

joblib.dump(eval_df, os.path.join(dir, f"{prefix}_eval_df.pkl"))
joblib.dump(user_dps_df, os.path.join(dir, f"{prefix}_user_dps_df.pkl"))
joblib.dump(feature_engineer, os.path.join(dir, f"{prefix}_feature_engineer.pkl"))

['artifacts/(mt)ngcf_v2_k5_1000_feature_engineer.pkl']

In [23]:
# import os
# import sys
# import joblib

# sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# dir = "artifacts"
# prefix = "(mt)ngcf_v2_k5_replica"
# eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))
# user_dps_df = joblib.load(os.path.join(dir, f"{prefix}_user_dps_df.pkl"))
# feature_engineer = joblib.load(os.path.join(dir, f"{prefix}_feature_engineer.pkl"))

### MMR

In [25]:
from post_processing.mmr import MMR

mmr_reranker = MMR(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
)


Seed set to 42


In [26]:
mmr_result_df = mmr_reranker.rerank(
    top_k=10,
    theta=0.5,
    random_state=42,
)
mmr_result_df.head()

Preparing input DataFrame for MMR...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [00:14<00:00, 143.60it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[356, 4993, 1227, 588, 318, 5418, 2467, 515, 6...","[356, 318, 3147, 4993, 5418, 588, 1213, 55765,...","[45722, 1233, 110, 2959, 2571]"
1,78,"[55290, 30749, 46723, 50872, 27803, 4011, 4993...","[55290, 30749, 50872, 46723, 4878, 46578, 5312...","[4119, 6993, 8400, 50872]"
2,127,"[42723, 48032, 55118, 46850, 41716, 8477, 3640...","[42723, 39234, 48032, 2067, 46850, 8491, 4690,...","[45726, 6958]"
3,170,"[296, 1348, 1090, 1291, 3787, 2686, 4011, 541,...","[296, 541, 1291, 1348, 1200, 1965, 260, 2353, ...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[46976, 5995, 27700, 34405, 342, 2677, 6786, 1...","[46976, 7387, 34405, 27700, 5995, 2174, 342, 1...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."


In [27]:
from common.eval import Evaluator
evaluator = Evaluator()

mmr_reranked_score_df = evaluator.evaluate(mmr_result_df, K=5)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=10)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=20)
mmr_reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.199663,0.036952,0.079360,0.246902,0.063340,0.073110,0.246902,0.063340,0.036555
std,20797.975208,0.322369,0.089494,0.134573,0.304543,0.110805,0.100261,0.304543,0.110805,0.050130
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.430677,0.041667,0.200000,0.440119,0.090909,0.100000,0.440119,0.090909,0.050000
max,71534.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.500000


In [28]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=mmr_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 723.96it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.118883,0.842902,0.122896,0.857885,0.485641
std,595.969798,0.069632,0.097329,0.097961,0.073390,0.057818
min,0.000000,0.000000,0.389671,0.000000,0.443311,0.284440
25%,515.750000,0.061924,0.780046,0.044164,0.822668,0.446503
50%,1031.500000,0.117415,0.857067,0.113013,0.874176,0.488540
75%,1547.250000,0.168731,0.920005,0.187570,0.909466,0.527102
max,2063.000000,0.351812,0.998682,0.552096,0.978108,0.658982


### DPA-RS

In [29]:
from post_processing.dpa_rs import DPA_RS

reranker = DPA_RS(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    ground_truth_dps_df=user_dps_df
)


Seed set to 42


In [30]:
result_df = reranker.rerank(
    top_k=10,
    max_iter=100,
    random_state=42,
)
result_df.head()

Preparing input DataFrame for DPA-RS...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Combined DataFrame shape: (206400, 16)
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [09:46<00:00,  3.52it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[45, 4232, 653, 27728, 733, 6537, 5379, 27801,...","[356, 318, 3147, 4993, 5418, 588, 1213, 55765,...","[45722, 1233, 110, 2959, 2571]"
1,78,"[46578, 27660, 4878, 5015, 1129, 1348, 4757, 1...","[55290, 30749, 50872, 46723, 4878, 46578, 5312...","[4119, 6993, 8400, 50872]"
2,127,"[1556, 42723, 49822, 7157, 6485, 5128, 40583, ...","[42723, 39234, 48032, 2067, 46850, 8491, 4690,...","[45726, 6958]"
3,170,"[296, 761, 2320, 7036, 3787, 1265, 5400, 4638,...","[296, 541, 1291, 1348, 1200, 1965, 260, 2353, ...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[1943, 33817, 8917, 37382, 53996, 52375, 7454,...","[46976, 7387, 34405, 27700, 5995, 2174, 342, 1...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."


In [31]:
from common.eval import Evaluator
evaluator = Evaluator()

reranked_score_df = evaluator.evaluate(result_df, K=5)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=10)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=20)
reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.124911,0.020349,0.046512,0.173248,0.039984,0.046124,0.173248,0.039984,0.023062
std,20797.975208,0.263911,0.060298,0.096710,0.263161,0.084974,0.070827,0.263161,0.084974,0.035414
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.333333,0.050321,0.100000,0.333333,0.050321,0.050000
max,71534.000000,1.000000,1.000000,0.600000,1.000000,1.000000,0.500000,1.000000,1.000000,0.250000


In [32]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 711.63it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.288993,0.978279,0.447585,0.932333,0.661797
std,595.969798,0.088663,0.022453,0.140372,0.031109,0.049487
min,0.000000,0.000000,0.697714,0.000000,0.722877,0.470101
25%,515.750000,0.233164,0.973491,0.373552,0.916036,0.632532
50%,1031.500000,0.290665,0.984307,0.457971,0.937621,0.665305
75%,1547.250000,0.346729,0.991830,0.537976,0.954325,0.694096
max,2063.000000,0.649125,1.000000,0.958373,0.986601,0.826162
